In [7]:
from flask import Flask, render_template, request
import re
import pickle
from nltk.stem import PorterStemmer
from spacy.lang.en.stop_words import STOP_WORDS
import spacy

app = Flask(__name__)
nlp = spacy.load('en_core_web_sm')

# Load summarization model (replace with your pickle file or HF pipeline)
with open('model.pkl', 'rb') as f:
    summarizer = pickle.load(f)

def nlp_pipeline(txt):
    text = re.sub(r"\s+", " ", txt)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9.,!?\s]", "", text)

    doc = nlp(text)
    data = [i for i in doc if i.text.lower() not in STOP_WORDS and i.text.isalpha()]
    ps = PorterStemmer()

    stems = [ps.stem(i.text) for i in data]
    lemmas = [i.lemma_ for i in data]
    pos = [(i.text, i.pos_) for i in data]
    ner = [(ent.text, ent.label_) for ent in doc.ents]

    return stems, lemmas, pos, ner, text

@app.route('/', methods=['GET', 'POST'])
def home():
    if request.method == 'POST':
        user_input = request.form['user_input']
        stems, lemmas, pos, ner, cleaned_text = nlp_pipeline(user_input)
        try:
            summary = summarizer(cleaned_text, max_length=100, min_length=30, do_sample=False)[0]['summary_text']
        except Exception as e:
            summary = f"Error generating summary: {e}"

        return render_template('index.html', 
                               cleaned=cleaned_text,
                               stems=stems,
                               lemmas=lemmas,
                               pos=pos,
                               ner=ner,
                               summary=summary)
    return render_template('index.html')
app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
Your max_length is set to 100, but your input_length is only 66. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=33)
Your max_length is set to 100, but your input_length is only 66. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=33)
127.0.0.1 - - [07/Nov/2025 22:01:36] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [07/Nov/2025 22:01:38] "POST / HTTP/1.1" 200 -
Your max_length is set to 100, but your input_length is only 83. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=41)
127.0.0.1 - - [07/Nov/2025 22:03:47] "POST / HTTP/1.1" 200 -
127.0.0.1 - -